# UJIAN TENGAH SEMESTER (UTS)
## PEMROGRAMAN SAINS DATA II (DS-2313)
**Institut Teknologi Sains Bandung (ITSB)**  
**Fakultas Digital, Desain dan Bisnis | Program Studi Sains Data**

| Atribut | Keterangan |
| :--- | :--- |
| **Mata Kuliah / Kode** | Pemrograman Sains Data II / DS-2313 (3 SKS: T = 2, P = 1) |
| **Dosen Pengampu** | Bakti Siregar, S.Si., M.Sc |
| **Semester / Tahun Akademik** | Semester 4 / TA Ganjil 2026/2027 |
| **Hari / Tanggal** | Rabu, 11 November 2026 |
| **Waktu** | $150 \text{ menit } (1 \times 3 \times 50')$ |
| **Sifat Ujian** | Kerjakan Mandiri dan Presentasikan di Video YouTube (Maksimum 10 Menit) |
| **Nama / NIM** | - |
| **Kelas** | - |

## Persiapan Lingkungan dan Dataset Ujian
Sel berikut memuat pustaka yang dibutuhkan, mendefinisikan fungsi utilitas pengujian (`uji` dan `jalankan_uji`), serta membangkitkan dataset ujian `hd` dan tabel referensi `klinik` sesuai spesifikasi soal (**tidak mengubah logika persiapan soal**).

In [2]:
import re
import numpy as np
import pandas as pd

# Utilitas pengujian publik (uji & jalankan_uji)
_DAFTAR_UJI = []

def uji(kondisi, pesan="Uji gagal"):
    """Memeriksa satu asersi pengujian publik dan mencatat hasilnya."""
    if not kondisi:
        raise AssertionError(f"[GAGAL] {pesan}")
    return True

def jalankan_uji(nama_bagian, daftar_fungsi_uji):
    """Menjalankan sekumpulan uji publik untuk suatu bagian soal."""
    lulus = 0
    total = len(daftar_fungsi_uji)
    print(f"=== Menjalankan Uji Publik {nama_bagian} ({total} uji) ===")
    for idx, (deskripsi, fn) in enumerate(daftar_fungsi_uji, start=1):
        try:
            fn()
            lulus += 1
            print(f"  [LULUS {idx}/{total}] {deskripsi}")
        except Exception as e:
            print(f"  [GAGAL {idx}/{total}] {deskripsi} -> {type(e).__name__}: {e}")
            raise
    print(f"Hasil {nama_bagian}: {lulus}/{total} uji publik LULUS.\n")

# Pembuatan data ujian (JANGAN DIUBAH)
URL = 'https://raw.githubusercontent.com/kb22/Heart-Disease-Prediction/master/dataset.csv'
asli = pd.read_csv(URL)

rng = np.random.default_rng(2026)
hd = asli.copy()
hd.insert(0, 'id_pasien', [f'P{i:04d}' for i in range(1, len(hd) + 1)])
hd['kode_klinik'] = rng.choice(['KL01', 'KL02', 'KL03', 'KL04', 'KL05'], len(hd))
hd['sex'] = hd['sex'].map({1: 'Laki-laki', 0: 'Perempuan'})
hd.loc[rng.choice(len(hd), 12, replace=False), 'sex'] = 'L'          # penulisan tidak konsisten
hd.loc[rng.choice(len(hd), 6, replace=False), 'sex'] = ' perempuan ' # huruf & spasi tidak konsisten
hd.loc[rng.choice(len(hd), 18, replace=False), 'chol'] = 0           # nilai mustahil (kode hilang)
hd.loc[rng.choice(len(hd), 15, replace=False), 'trestbps'] = np.nan  # nilai hilang
hd.loc[rng.choice(len(hd), 2, replace=False), 'chol'] = 5640         # galat input
hd['age'] = hd['age'].astype(str) + 'th'                             # angka sebagai teks
hd = pd.concat([hd, hd.sample(6, random_state=3)], ignore_index=True) # duplikat

klinik = pd.DataFrame({
    'kode_klinik': ['KL01', 'KL02', 'KL03', 'KL04', 'KL06'],
    'kota': ['Bekasi', 'Bandung', 'Jakarta', 'Bogor', 'Depok'],
    'tipe': ['RS Swasta', 'RSUD', 'RS Swasta', 'Klinik Utama', 'RSUD']
})

HD_ASLI = hd.copy()  # untuk memeriksa bahwa fungsi tidak mengubah masukan
print('hd:', hd.shape, '| klinik:', klinik.shape)  # hd: (309, 16) | klinik: (5, 3)

hd: (309, 16) | klinik: (5, 3)


---
## BAGIAN A — KONSEP
**20 poin | A1–A8 @ 2 poin, A9 @ 4 poin | CPL-1 | CPMK-1, CPMK-2**

### Soal A1 — Broadcasting dan Memori Array NumPy *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-1)*
> **Pertanyaan:** Tuliskan aturan *broadcasting* NumPy sebagai algoritma (pseudocode) yang menerima dua *shape* dan mengembalikan *shape* hasil atau galat. Apa hasil `(4, 3)` dengan `(3,)` dan `(4, 3)` dengan `(4,)`? Bagaimana program dapat mendeteksi bahwa sebuah potongan array adalah *view*?

#### Jawaban A1:

**1. Algoritma (Pseudocode) Aturan *Broadcasting* NumPy**

NumPy membandingkan dimensi dua array secara elemen-per-elemen **mulai dari dimensi paling kanan (belakang) menuju ke kiri**. Jika jumlah dimensi (*ndim*) berbeda, *shape* yang lebih pendek disisipi (*prepend*) nilai `1` di sebelah kiri hingga panjang kedua *shape* sama.

```text
ALGORITMA HitungShapeBroadcast(shape_A, shape_B):
    INPUT : shape_A (tuple bilangan bulat), shape_B (tuple bilangan bulat)
    OUTPUT: shape_hasil (tuple bilangan bulat) atau GALAT (ValueError)

    1. n_A <- panjang(shape_A)
    2. n_B <- panjang(shape_B)
    3. n_maks <- MAKS(n_A, n_B)

    4. // Ratakan kanan (right-align) dengan menyisipkan 1 di sebelah kiri
       A_pad <- (1 sebanyak (n_maks - n_A) kali) digabung dengan shape_A
       B_pad <- (1 sebanyak (n_maks - n_B) kali) digabung dengan shape_B

    5. shape_hasil <- daftar kosong []

    6. // Periksa kompatibilitas dari dimensi paling kanan ke kiri (atau indeks 0..n_maks-1 setelah dipad)
       UNTUK i dari (n_maks - 1) turun ke 0 LAKUKAN:
           d_A <- A_pad[i]
           d_B <- B_pad[i]

           JIKA d_A == d_B MAKA:
               Masukkan d_A ke awal shape_hasil
           JIKA TIDAK, JIKA d_A == 1 MAKA:
               Masukkan d_B ke awal shape_hasil   // dimensi A diregangkan mengikuti B
           JIKA TIDAK, JIKA d_B == 1 MAKA:
               Masukkan d_A ke awal shape_hasil   // dimensi B diregangkan mengikuti A
           JIKA TIDAK:
               LEMPAR GALAT ValueError("Operands could not be broadcast together")

    7. KEMBALIKAN tuple(shape_hasil)
```

---

**2. Penelusuran Hasil *Broadcasting***

* **Kasus 1: `(4, 3)` dengan `(3,)`**
  1. Penyamaan panjang dimensi (*left-padding* dengan `1`):
     * `shape_A = (4, 3)`
     * `shape_B = (1, 3)`
  2. Pemeriksaan dari kanan ke kiri:
     * Dimensi ke-1 (kanan): `3` dan `3` $\rightarrow$ **cocok**, hasil dimensi = `3`.
     * Dimensi ke-0 (kiri): `4` dan `1` $\rightarrow$ **kompatibel** (salah satu bernilai `1`), diregangkan menjadi `4`.
  3. **Hasil akhir:** **`(4, 3)`** (berhasil).

* **Kasus 2: `(4, 3)` dengan `(4,)`**
  1. Penyamaan panjang dimensi (*left-padding* dengan `1`):
     * `shape_A = (4, 3)`
     * `shape_B = (1, 4)`
  2. Pemeriksaan dari kanan ke kiri:
     * Dimensi ke-1 (kanan): `3` dan `4` $\rightarrow$ **tidak cocok** ($3 \neq 4$ dan tidak ada yang bernilai `1`).
  3. **Hasil akhir:** **Galat (`ValueError: operands could not be broadcast together with shapes (4,3) (4,)`)**. Agar kompatibel, array kedua harus di-*reshape* secara eksplisit menjadi `(4, 1)`.

---

**3. Cara Program Mendeteksi Apakah Potongan Array adalah *View***

Sebuah *view* tidak memiliki penyangga memori (*data buffer*) sendiri, melainkan merujuk langsung pada memori array asal. Program dapat mendeteksinya melalui tiga cara utama di NumPy:

1. **Memeriksa atribut `.base`**:
   * Jika `potongan.base is not None` (atau secara spesifik `potongan.base is arr_asli`), maka `potongan` adalah **view** (objek meminjam memori dari objek di `.base`). Jika `potongan.base is None`, maka array tersebut memiliki datanya sendiri (*copy* atau array asli).
2. **Memeriksa bendera kepemilikan memori (`flags.owndata`)**:
   * `potongan.flags.owndata` bernilai `False` untuk *view* dan `True` untuk *copy*.
3. **Menggunakan fungsi `np.shares_memory(arr_asli, potongan)`**:
   * Mengembalikan `True` jika kedua array berbagi blok memori fisik yang tumpang tindih di RAM.

### Soal A2 — Algoritma *Hash Join* pada `merge` dan Bahaya Kunci Ganda *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-1)*
> **Pertanyaan:** Jelaskan algoritma *hash join* di balik `merge` beserta kompleksitasnya, dibandingkan *nested loop join*. Mengapa *join* dengan kunci ganda di tabel kanan berbahaya, dan bagaimana program mendeteksinya sebelum *join* dijalankan?

#### Jawaban A2:

**1. Algoritma *Hash Join* vs *Nested Loop Join* beserta Kompleksitasnya**

Misalkan tabel kiri $L$ memiliki $M$ baris dan tabel kanan $R$ memiliki $N$ baris:

* **Algoritma *Hash Join* (digunakan oleh `pd.merge`):**
  1. **Fase *Build* (Membangun Tabel Hash):** Memindai satu kali tabel yang lebih kecil (atau tabel kanan $R$) untuk menghitung nilai fungsi hash $h(k)$ dari setiap kunci join $k \in R$, lalu menyimpan pasangan `(kunci, indeks_baris_R)` ke dalam ember (*hash bucket*). Kompleksitas waktu rata-rata: $\mathcal{O}(N)$.
  2. **Fase *Probe* (Pencarian Pasangan):** Memindai tabel kiri $L$ baris demi baris, menghitung $h(k)$ untuk setiap kunci di $L$, dan mencarinya di tabel hash dengan waktu pencarian konstan $\mathcal{O}(1)$ rata-rata. Kompleksitas waktu rata-rata: $\mathcal{O}(M)$.
  * **Kompleksitas Total *Hash Join*:** Waktu rata-rata **$\mathcal{O}(M + N)$** dan memori tambahan **$\mathcal{O}(N)$** (untuk menyimpan *hash table*).
* **Perbandingan dengan *Nested Loop Join*:**
  * *Nested loop join* membandingkan setiap baris di $L$ terhadap seluruh baris di $R$ menggunakan dua perulangan bersarang.
  * **Kompleksitas *Nested Loop Join*:** Waktu **$\mathcal{O}(M \times N)$** (kuadratik) dengan memori tambahan $\mathcal{O}(1)$. Untuk $M, N = 10^5$, *nested loop* membutuhkan $10^{10}$ operasi (sangat lambat), sedangkan *hash join* hanya sekitar $2 \times 10^5$ operasi.

---

**2. Mengapa *Join* dengan Kunci Ganda di Tabel Kanan Berbahaya?**

Pada skenario pengayaan data (*left join* / *many-to-one*, misalnya menggabungkan tabel pasien dengan tabel referensi klinik):
1. **Ledakan Baris (*Row Multiplication / Fan-out*) Tanpa Pesan Galat:** Jika satu `kode_klinik` muncul $k$ kali di tabel referensi kanan, maka setiap pasien dari klinik tersebut akan digandakan menjadi $k$ baris pada tabel hasil *merge*. Pandas tidak melempar error secara *default*.
2. **Distorsi Statistik dan Agregasi Hilir (*Silent Data Corruption*):** Duplikasi baris yang tidak disadari akan menggelembungkan jumlah pasien (`count`), menggeser nilai rata-rata (`mean`), merusak distribusi target, dan menyebabkan **kebocoran data (*data leakage*)** apabila baris ganda dari pasien yang sama terpisah ke dalam *train set* dan *test set*.

---

**3. Cara Program Mendeteksinya Sebelum *Join* Dijalankan**

Program dapat menerapkan *defensive programming* sebelum atau saat memanggil `merge`:
1. **Pemeriksaan Eksplisit Sebelum Join:**
   ```python
   if kanan[kunci].duplicated().any():  # atau: if not kanan[kunci].is_unique:
       raise ValueError(f"Kunci kanan '{kunci}' mengandung duplikat!")
   ```
2. **Menggunakan Parameter `validate` pada `pd.merge`:**
   ```python
   hasil = pd.merge(kiri, kanan, on=kunci, how='left', validate='many_to_one')  # atau 'm:1'
   ```
   Parameter ini memerintahkan Pandas memverifikasi keunikan kunci di tabel kanan terlebih dahulu dan langsung melempar `pd.errors.MergeError` bila ditemukan kunci ganda.

### Soal A3 — Mekanisme Data Hilang (MCAR, MAR, MNAR) *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-2)*
> **Pertanyaan:** Jelaskan MCAR, MAR, dan MNAR dengan contoh rekam medis. Tuliskan algoritma sederhana untuk menguji secara otomatis apakah nilai hilang suatu kolom konsisten dengan MCAR. Mengapa MNAR tidak dapat diuji dari data?

#### Jawaban A3:

**1. Penjelasan MCAR, MAR, dan MNAR beserta Contoh Rekam Medis**

Misalkan $Y$ adalah variabel yang memiliki nilai hilang, $R \in \{0, 1\}$ adalah indikator kehilangan ($R=1$ jika $Y$ hilang), $X_{\text{obs}}$ adalah variabel lain yang teramati di dataset, dan $Y_{\text{mis}}$ adalah nilai $Y$ yang hilang itu sendiri:

1. **MCAR (*Missing Completely at Random*)** — $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}}) = P(R = 1)$:
   * Peluang suatu nilai hilang murni acak dan **tidak bergantung** pada variabel teramati mana pun maupun pada nilai yang hilang itu sendiri.
   * *Contoh Rekam Medis:* Hasil kolesterol (`chol`) beberapa pasien hilang karena tabung sampel darah tidak sengaja terjatuh di laboratorium atau gangguan listrik sesaat pada alat pengukur tekanan darah (`trestbps`).
2. **MAR (*Missing at Random*)** — $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}}) = P(R = 1 \mid X_{\text{obs}})$:
   * Peluang nilai hilang bergantung secara sistematis pada **variabel lain yang tercatat/teramati ($X_{\text{obs}}$)**, tetapi tidak bergantung pada nilai $Y_{\text{mis}}$ setelah $X_{\text{obs}}$ dikontrol.
   * *Contoh Rekam Medis:* Dokter lebih jarang memerintahkan tes kolesterol lengkap (`chol`) pada pasien berusia muda (`age < 30`) yang tidak memiliki keluhan nyeri dada (`cp = 0`). Di sini, hilangnya `chol` bergantung pada `age` dan `cp` yang nilainya tercatat lengkap di rekam medis.
3. **MNAR (*Missing Not at Random*)** — $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}})$ masih bergantung pada $Y_{\text{mis}}$:
   * Peluang nilai hilang bergantung langsung pada **nilai yang tidak teramati itu sendiri** (atau faktor laten yang tidak diukur).
   * *Contoh Rekam Medis:* Pasien dengan depresi segmen ST (`oldpeak`) atau detak jantung latihan (`thalach`) yang sangat buruk gagal menyelesaikan uji *treadmill* karena kelelahan parah sehingga nilainya kosong; atau pasien perokok berat/obesitas ekstrem sengaja mengosongkan kuesioner berat badan karena enggan mengakuinya.

---

**2. Algoritma Sederhana Pengujian Otomatis Konsistensi MCAR**

Jika data berstatus MCAR, maka distribusi setiap kolom teramati $X_j$ tidak boleh berbeda secara signifikan antara kelompok baris yang $Y$-nya hilang ($R=1$) dan kelompok baris yang $Y$-nya lengkap ($R=0$).

```text
ALGORITMA UjiKonsistensiMCAR(df, kolom_target, alpha = 0.05):
    INPUT : DataFrame df, nama kolom_target yang memiliki NaN, taraf nyata alpha
    OUTPUT: Status ("Konsisten MCAR" atau "Tolak MCAR / Indikasi MAR") beserta daftar kolom terkait

    1. R <- indikator biner (1 jika df[kolom_target] bernilai NaN, 0 jika tidak)
    2. JIKA jumlah(R == 1) == 0 ATAU jumlah(R == 0) == 0 MAKA:
           KEMBALIKAN "Tidak dapat diuji (tidak ada variasi nilai hilang)"

    3. kolom_lain <- semua kolom di df selain kolom_target dan kolom ID
    4. alpha_koreksi <- alpha / panjang(kolom_lain)   // Koreksi Bonferroni untuk uji ganda
    5. kolom_signifikan <- daftar kosong []

    6. UNTUK SETIAP kol DI kolom_lain LAKUKAN:
           JIKA kol bertipe numerik MAKA:
               grup_hilang <- df.loc[R == 1, kol].dropna()
               grup_lengkap <- df.loc[R == 0, kol].dropna()
               // Bandingkan lokasi sebaran dengan Mann-Whitney U test atau Welch's t-test
               p_val <- UjiMannWhitneyU(grup_hilang, grup_lengkap)
           JIKA TIDAK (kol bertipe kategorikal) MAKA:
               tabel_kontingensi <- pd.crosstab(df[kol].fillna("MISSING"), R)
               // Uji independensi Chi-Square
               p_val <- UjiChiSquare(tabel_kontingensi)

           JIKA p_val < alpha_koreksi MAKA:
               Tambahkan (kol, p_val) ke kolom_signifikan

    7. JIKA kolom_signifikan TIDAK KOSONG MAKA:
           KEMBALIKAN ("Tolak MCAR (Indikasi MAR)", kolom_signifikan)
       JIKA TIDAK:
           KEMBALIKAN ("Konsisten dengan MCAR", [])
```

---

**3. Mengapa MNAR Tidak Dapat Diuji dari Data Semata?**

Untuk membuktikan atau menolak MNAR, kita harus menguji apakah $P(R = 1 \mid X_{\text{obs}}, Y_{\text{mis}}) \neq P(R = 1 \mid X_{\text{obs}})$, yang berarti membandingkan distribusi nilai $Y$ pada grup yang teramati ($R=0$) dengan distribusi nilai $Y$ pada grup yang **hilang** ($R=1$). Karena nilai $Y_{\text{mis}}$ pada grup $R=1$ **tidak pernah tercatat di dalam dataset**, informasi empiris untuk menghitung distribusi tersebut tidak ada. Tanpa data eksternal (seperti pelacakan ulang / audit rekam medis fisik terhadap pasien yang datanya kosong), ketergantungan pada nilai yang hilang bersifat *unidentifiable* dari data observasi saja.

### Soal A4 — Deteksi Kode Hilang dan Kebijakan Konsensus *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-2)*
> **Pertanyaan:** Pada Pertemuan 3, aturan *"lonjakan frekuensi di ujung distribusi"* dipakai untuk mendeteksi kode hilang. Jelaskan logikanya. Mengapa `Pregnancies = 0` tidak ditandai, sedangkan `Insulin = 0` ditandai? Mengapa pembersih otomatis memakai kebijakan konsensus ($\ge 2$ aturan) sebelum mengubah data?

#### Jawaban A4:

**1. Logika Aturan *"Lonjakan Frekuensi di Ujung Distribusi"* (*Boundary Spike*)**

Pada sistem pencatatan lama atau formulir klinis, nilai yang tidak diukur sering diisi dengan **nilai sentinel (kode hilang)** berupa angka khusus di luar rentang normal—paling sering `0`, `-1`, atau `999`. Secara statistik:
* Variabel kontinu alami (seperti kolesterol, tekanan darah, atau kadar insulin) memiliki fungsi kepadatan peluang yang **melandai secara mulus (*smooth tail*)** saat mendekati batas bawah maupun batas atas.
* Apabila seluruh nilai hilang dikodekan sebagai satu angka konstan (misalnya `0`), maka seluruh massa probabilitas dari data yang hilang akan menumpuk di satu titik ekstrem tersebut. Akibatnya, terbentuk **dua anomali struktural sekaligus**:
  1. **Lonjakan frekuensi (*frequency spike*):** Frekuensi nilai `0` jauh lebih tinggi daripada nilai-nilai kecil di dekatnya.
  2. **Celah distribusi (*distribution gap*):** Terdapat jarak kosong yang lebar (misalnya $> 0.5 \times \text{IQR}$) antara nilai `0` dengan nilai terukur terkecil berikutnya.

---

**2. Mengapa `Pregnancies = 0` Tidak Ditandai, Sedangkan `Insulin = 0` Ditandai?**

* **`Pregnancies = 0` (Nilai Sah / Valid):**
  1. *Kontinuitas Sebaran (Tanpa Celah):* `Pregnancies` adalah variabel cacah diskrit ($0, 1, 2, 3, \dots$). Nilai `0` bersebelahan langsung dengan `1, 2, 3` tanpa *gap* ekstrem terhadap $\text{IQR}$, dan penurunan frekuensinya dari $0 \to 1 \to 2$ mengikuti pola distribusi cacah alami (seperti Poisson/Negatif Binomial).
  2. *Makna Domain:* Pasien yang belum pernah hamil secara faktual memiliki jumlah kehamilan `0`.
* **`Insulin = 0` (Kode Hilang / Mustahil):**
  1. *Disrupsi & Celah Sebaran:* Setelah nilai `0`, nilai insulin positif terkecil berikutnya melompat jauh (misal $\ge 14\text{ }\mu\text{U/mL}$), menciptakan celah besar terhadap $\text{IQR}$ disertai lonjakan ratusan baris di angka `0`.
  2. *Makna Fisiologis:* Kadar insulin serum `0` mutlak pada manusia hidup secara medis mustahil; angka `0` tersebut adalah kode pengganti untuk pemeriksaan lab yang tidak dilakukan.

---

**3. Mengapa Pembersih Otomatis Memakai Kebijakan Konsensus ($\ge 2$ Aturan)?**

Mengubah nilai data asli menjadi `NaN` adalah operasi destruktif bila salah sasaran (*false positive*). Jika pembersih otomatis hanya mengandalkan **satu aturan tunggal**, risiko kesalahan sangat tinggi:
* Jika hanya memakai aturan *"frekuensi nilai 0 tinggi"*, maka variabel seperti **`oldpeak = 0`** (depresi ST saat istirahat, di mana pasien normal memang bernilai `0.0`) atau **`Pregnancies = 0`** akan ikut terhapus secara keliru.
* Sebaliknya, jika menggabungkan **konsensus $\ge 2$ aturan independen**—misalnya: **(Aturan 1)** kolom bersifat kontinu (banyak nilai unik), **(Aturan 2)** frekuensi di titik `0` melonjak dibanding tetangga, **DAN (Aturan 3)** terpisah oleh celah $> 0.5 \cdot \text{IQR}$ dari nilai positif terkecil berikutnya—maka `oldpeak = 0` aman (karena menyatu mulus dengan `0.1, 0.2, ...` tanpa celah), sedangkan `chol = 0` (yang melompat dari `0` langsung ke $> 120\text{ mg/dL}$) terdeteksi secara presisi.

### Soal A5 — Kontrak `fit/transform` pada *Transformer* Data *(2 poin | CPL-1 | CPMK-1 | Sub-CPMK-2)*
> **Pertanyaan:** Jelaskan kontrak `fit/transform`. Mengapa transformasi yang mempelajari statistik harus berupa kelas, bukan fungsi? Apa arti konvensi atribut berakhiran `_` dan mengapa `fit` mengembalikan `self`? Apa yang harus terjadi bila `transform` dipanggil sebelum `fit`?

#### Jawaban A5:

**1. Kontrak `fit/transform`**

Kontrak `fit/transform` (dipopulerkan oleh *scikit-learn*) memisahkan proses pra-pemrosesan data menjadi dua tahap yang tegas:
* **`fit(X, y=None)`**: Mempelajari parameter atau statistik distribusi **hanya dari data latih (`X_train`)**—seperti median, $\text{IQR}$, rata-rata, simpangan baku, atau himpunan kategori unik—lalu menyimpannya di dalam objek tanpa mengubah data.
* **`transform(X)`**: Menggunakan parameter yang telah disimpan dari tahap `fit` untuk mengubah data masukan (`X_train`, `X_val`, maupun `X_test`) secara konsisten dan mengembalikan objek data baru.

---

**2. Mengapa Harus Berupa Kelas (*Class*), Bukan Fungsi Biasa?**

Transformasi yang bergantung pada statistik data bersifat **berkeadaan (*stateful*)**:
* Jika ditulis sebagai fungsi biasa `skalakan(df)` yang langsung menghitung median dan $\text{IQR}$ dari argumen yang diberikan, maka saat fungsi tersebut dipanggil pada `X_test`, ia akan menghitung median dari `X_test` itu sendiri (menyebabkan **kebocoran data / *data leakage*** dan inkonsistensi skala ketika `X_test` hanya berisi 1 baris di lingkungan produksi).
* Jika fungsi dipaksa mengembalikan tuple `(df_tertransformasi, dict_parameter)`, pengguna harus mengelola dan meneruskan kamus parameter secara manual untuk setiap langkah pipeline.
* Dengan **kelas (OOP)**, parameter statistik dikapsulasi sebagai *state* internal objek (`self.median_`, `self.iqr_`), sehingga objek yang sudah di-`fit` pada data latih dapat langsung memanggil `.transform(X_test)` kapan saja dengan skala yang dijamin identik.

---

**3. Arti Konvensi Atribut Berakhiran `_` dan Mengapa `fit` Mengembalikan `self`**

* **Atribut berakhiran garis bawah tunggal `_` (misalnya `self.median_`, `self.iqr_`, `self.kategori_`)**:
  * Menandakan bahwa atribut tersebut adalah **atribut hasil pembelajaran (*fitted attribute*)** yang dihitung dari data saat `.fit()` dijalankan, untuk membedakannya secara jelas dari hiperparameter/konfigurasi awal yang diberikan pengguna saat inisialisasi `__init__()` (seperti `self.kolom`).
  * Keberadaan atribut berakhiran `_` juga menjadi penanda standar untuk memeriksa apakah objek sudah pernah di-`fit` (`hasattr(self, 'median_')`).
* **Mengapa `fit(X)` mengembalikan `self` (`return self`)**:
  * Memungkinkan **perantaian metode (*method chaining*)** yang ringkas dan ekspresif dalam satu baris, seperti:
    ```python
    X_latih_skala = SkalaRobustku(kolom_num).fit(X_latih).transform(X_latih)
    ```

---

**4. Apa yang Harus Terjadi Bila `transform` Dipanggil Sebelum `fit`?**

Metode `transform(X)` **wajib menolak eksekusi dan langsung melempar eksepsi `RuntimeError`** (atau `NotFittedError` pada *scikit-learn*) dengan pesan yang jelas. Hal ini mencegah program berjalan dengan atribut yang belum terdefinisi (`AttributeError` yang membingungkan) atau menggunakan nilai *default* sembarang yang merusak validitas fitur.

### Soal A6 — *Gradient Descent* Adaptif dan Metrik Regresi *(2 poin | CPL-1 | CPMK-2 | Sub-CPMK-3)*
> **Pertanyaan:** Tuliskan pseudocode *gradient descent* untuk regresi linear yang (a) memiliki kriteria henti dan (b) mendeteksi divergensi serta memperbaikinya sendiri. Tuliskan rumus $R^2$, MAE, RMSE dan jelaskan mengapa $\text{RMSE} \ge \text{MAE}$.

#### Jawaban A6:

**1. Pseudocode *Gradient Descent* Regresi Linear dengan Kriteria Henti & Perbaikan Divergensi Otomatis**

Misalkan fungsi objektif adalah *Mean Squared Error* $J(w) = \frac{1}{2n} \|X w - y\|_2^2$ dengan gradien $\nabla J(w) = \frac{1}{n} X^T (X w - y)$:

```text
ALGORITMA GradientDescentRegresiLinear(X, y, lr = 0.1, tol = 1e-6, max_iter = 10000):
    INPUT : Matriks fitur X (n x p, sudah ditambah kolom 1 untuk intersep),
            vektor target y (n x 1), laju belajar awal lr, toleransi tol, batas iterasi max_iter
    OUTPUT: Vektor bobot optimal w (p x 1)

    1. n, p <- dimensi(X)
    2. w <- vektor nol berukuran (p x 1)
    3. loss_lama <- (1 / (2 * n)) * jumlah((X * w - y)^2)

    4. UNTUK iterasi = 1 SAMPAI max_iter LAKUKAN:
           galat <- (X * w) - y
           gradien <- (1 / n) * (transpose(X) * galat)

           // (a) Kriteria henti 1: Norma gradien sudah sangat mendekati nol
           JIKA norma_L2(gradien) < tol MAKA:
               BERHENTI (Konvergen)

           // Kandidat pembaruan bobot
           w_baru <- w - lr * gradien
           loss_baru <- (1 / (2 * n)) * jumlah((X * w_baru - y)^2)

           // (b) Deteksi divergensi (loss membesar atau menjadi NaN/Inf) & perbaikan mandiri
           JIKA tidak_berhingga(loss_baru) ATAU (loss_baru > loss_lama) MAKA:
               lr <- lr * 0.5                 // Pangkas learning rate separuhnya (backtracking)
               JIKA lr < 1e-15 MAKA:
                   BERHENTI                   // Mencegah loop tak hingga bila sudah di batas presisi
               LANJUT KE ITERASI BERIKUTNYA   // Tolak w_baru, pertahankan w lama
           
           // (a) Kriteria henti 2: Penurunan nilai loss sudah di bawah ambang toleransi
           JIKA |loss_lama - loss_baru| < tol MAKA:
               w <- w_baru
               BERHENTI (Konvergen)

           // Terima langkah pembaruan
           w <- w_baru
           loss_lama <- loss_baru

    5. KEMBALIKAN w
```

---

**2. Rumus Matematis $R^2$, MAE, dan RMSE**

Untuk target aktual $y = (y_1, \dots, y_n)$, prediksi $\hat{y} = (\hat{y}_1, \dots, \hat{y}_n)$, rata-rata $\bar{y} = \frac{1}{n}\sum_{i=1}^n y_i$, dan residual $e_i = y_i - \hat{y}_i$:

1. **Koefisien Determinasi ($R^2$):**
   $$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$
2. ***Mean Absolute Error* (MAE):**
   $$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i| = \frac{1}{n} \sum_{i=1}^{n} |e_i|$$
3. ***Root Mean Squared Error* (RMSE):**
   $$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} e_i^2}$$

---

**3. Mengapa $\text{RMSE} \ge \text{MAE}$ Selalu Berlaku?**

Pandang himpunan galat mutlak $|e_1|, |e_2|, \dots, |e_n|$ sebagai peubah acak diskrit berseragam dengan nilai harapan $\mathbb{E}[|e|] = \text{MAE}$ dan momen kedua $\mathbb{E}[|e|^2] = \text{RMSE}^2$. Berdasarkan definisi varians yang selalu non-negatif ($\text{Var}(|e|) \ge 0$) maupun **Pertidaksamaan Jensen** pada fungsi konveks $g(u) = u^2$:

$$\text{Var}(|e|) = \mathbb{E}\left[|e|^2\right] - \left(\mathbb{E}[|e|]\right)^2 = \text{RMSE}^2 - \text{MAE}^2 \ge 0$$

Sehingga diperoleh hubungan eksak:

$$\text{RMSE} = \sqrt{\text{MAE}^2 + \text{Var}(|e|)} \ge \text{MAE}$$

* **Kesimpulan:** $\text{RMSE}$ selalu lebih besar atau sama dengan $\text{MAE}$. Kesamaan ($\text{RMSE} = \text{MAE}$) hanya terjadi jika $\text{Var}(|e|) = 0$, yaitu ketika seluruh residual memiliki besar mutlak yang identik ($|e_1| = |e_2| = \dots = |e_n|$). Semakin ekstrem *outlier* pada residual, semakin besar $\text{Var}(|e|)$ dan semakin jauh $\text{RMSE}$ melampaui $\text{MAE}$.

### Soal A7 — Stabilitas Numerik *Sigmoid*, *Log-Loss*, dan Interpretasi Regresi Logistik *(2 poin | CPL-1 | CPMK-2 | Sub-CPMK-3)*
> **Pertanyaan:** Mengapa `1/(1+np.exp(-z))` dan `np.log(p)` dapat gagal secara numerik? Tuliskan versi stabil *sigmoid* dan *log-loss*. Apa arti $\exp(\beta_j)$ pada regresi logistik?

#### Jawaban A7:

**1. Mengapa `1 / (1 + np.exp(-z))` dan `np.log(p)` Dapat Gagal Secara Numerik?**

* **Kegagalan `1 / (1 + np.exp(-z))` (*Floating-Point Overflow*):**
  * Tipe data `float64` (IEEE 754) memiliki batas nilai maksimum sekitar $1.797 \times 10^{308}$, sehingga fungsi eksponensial `np.exp(x)` akan mengalami **overflow menjadi `inf`** apabila argumennya $x > 709.78$.
  * Ketika logit $z$ bernilai negatif besar (misalnya $z = -800$), maka $-z = +800 > 709.78$. Akibatnya, `np.exp(-z)` menghasilkan `inf` dan memicu `RuntimeWarning: overflow encountered in exp` (meskipun secara matematis $1/(1+\infty) = 0$).
* **Kegagalan `np.log(p)` dan `np.log(1 - p)` (*Log of Zero & NaN*):**
  * Ketika $|z|$ cukup besar (misal $z > 37$ atau $z < -710$), probabilitas hasil sigmoid membulat secara *floating-point* menjadi tepat $p = 1.0$ atau $p = 0.0$.
  * Memanggil `np.log(0.0)` menghasilkan `-inf`. Lebih parah lagi, pada rumus *binary cross-entropy* $-y \log(p) - (1-y)\log(1-p)$, ketika $y = 0$ dan $p = 0$, suku pertama menghitung `0 * (-inf)` yang menghasilkan **`NaN`** dan merusak seluruh perhitungan gradien.

---

**2. Versi Stabil Secara Numerik untuk *Sigmoid* dan *Log-Loss***

* **a. *Sigmoid* Stabil (Memastikan Eksponen Selalu $\le 0$):**
  Gunakan identitas aljabar $\frac{1}{1 + e^{-z}} = \frac{e^z}{1 + e^z}$ untuk memisahkan kasus $z \ge 0$ dan $z < 0$ sehingga argumen `np.exp` tidak pernah positif:
  $$\sigma(z) = \begin{cases} \dfrac{1}{1 + e^{-z}}, & z \ge 0 \\[8pt] \dfrac{e^{z}}{1 + e^{z}}, & z < 0 \end{cases}$$

  ```python
  def sigmoid_stabil(z):
      z = np.asarray(z, dtype=float)
      # Hindari evaluasi eksponen positif di cabang yang tidak dipilih
      z_aman = np.clip(z, -500.0, 500.0)
      return np.where(
          z >= 0,
          1.0 / (1.0 + np.exp(-z_aman)),
          np.exp(z_aman) / (1.0 + np.exp(z_aman))
      )
  ```

* **b. *Log-Loss* (*Binary Cross-Entropy*) Stabil:**
  1. **Jika masukan berupa probabilitas $p \in [0, 1]$** (gunakan *clipping* ke $[arepsilon, 1 - arepsilon]$ dengan $\varepsilon = 10^{-15}$):
     ```python
     def log_loss_dari_prob(y, p, eps=1e-15):
         p_aman = np.clip(p, eps, 1.0 - eps)
         return -np.mean(y * np.log(p_aman) + (1.0 - y) * np.log1p(-p_aman))
     ```
  2. **Jika masukan langsung berupa logit $z = Xw + b$** (substitusi analitik dengan `np.log1p` dan eksponen non-positif $-|z|$):
     $$\ell(y, z) = \max(z, 0) - y \cdot z + \ln\left(1 + e^{-|z|}\right)$$
     ```python
     def log_loss_dari_logit(y, z):
         return np.mean(np.maximum(z, 0.0) - y * z + np.log1p(np.exp(-np.abs(z))))
     ```

---

**3. Arti $\exp(\beta_j)$ pada Regresi Logistik**

Pada regresi logistik, model memprediksi *log-odds* (logit) kelas positif:
$$\ln\left(\frac{p}{1 - p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_j x_j + \dots + \beta_k x_k$$

Jika fitur $x_j$ bertambah $1$ satuan menjadi $x_j + 1$ sementara seluruh fitur lainnya tetap konstan (*ceteris paribus*), maka rasio antara *odds* baru dan *odds* awal adalah:
$$\frac{\text{Odds}(x_j + 1)}{\text{Odds}(x_j)} = \frac{\exp(\beta_0 + \dots + \beta_j(x_j + 1) + \dots)}{\exp(\beta_0 + \dots + \beta_j x_j + \dots)} = \exp(\beta_j)$$

Dengan demikian, **$\exp(\beta_j)$ adalah *Odds Ratio* (OR)**, yaitu faktor pengali (*multiplicative factor*) terhadap *odds* ($\frac{p}{1-p}$) terjadinya kejadian positif ($Y=1$) untuk setiap kenaikan 1 unit pada variabel prediktor $x_j$:
* Jika $\beta_j > 0 \implies \exp(\beta_j) > 1$: kenaikan $x_j$ **meningkatkan** *odds* penyakit jantung sebesar $(\exp(\beta_j) - 1) \times 100\%$.
* Jika $\beta_j < 0 \implies 0 < \exp(\beta_j) < 1$: kenaikan $x_j$ **menurunkan** *odds* penyakit jantung.
* Jika $\beta_j = 0 \implies \exp(\beta_j) = 1$: fitur $x_j$ tidak mengubah *odds*.

### Soal A8 — Perhitungan Metrik Klasifikasi dan Kompleksitas ROC-AUC *(2 poin | CPL-1 | CPMK-2 | Sub-CPMK-4)*
> **Pertanyaan:** Dari *confusion matrix* $TN=90$, $FP=10$, $FN=30$, $TP=70$ hitung akurasi, presisi, recall, dan F1. Jelaskan cara menghitung ROC-AUC tanpa menggambar kurva beserta kompleksitasnya.

#### Jawaban A8:

**1. Perhitungan Manual Akurasi, Presisi, Recall, dan F1-Score**

Diketahui komponen *confusion matrix*:
* $TN = 90$, $FP = 10$, $FN = 30$, $TP = 70$
* Total sampel keseluruhan: $N = TN + FP + FN + TP = 90 + 10 + 30 + 70 = 200$

1. **Akurasi (*Accuracy*):**
   $$\text{Akurasi} = \frac{TP + TN}{N} = \frac{70 + 90}{200} = \frac{160}{200} = \mathbf{0.80} \text{ } (\mathbf{80\%})$$

2. **Presisi (*Precision* / *Positive Predictive Value*):**
   $$\text{Presisi} = \frac{TP}{TP + FP} = \frac{70}{70 + 10} = \frac{70}{80} = \mathbf{0.875} \text{ } (\mathbf{87.5\%})$$

3. **Recall (*Sensitivity* / *True Positive Rate*):**
   $$\text{Recall} = \frac{TP}{TP + FN} = \frac{70}{70 + 30} = \frac{70}{100} = \mathbf{0.70} \text{ } (\mathbf{70\%})$$

4. **F1-Score (Rata-rata Harmonik Presisi dan Recall):**
   $$\text{F1} = \frac{2 \times \text{Presisi} \times \text{Recall}}{\text{Presisi} + \text{Recall}} = \frac{2 \times 0.875 \times 0.70}{0.875 + 0.70} = \frac{1.225}{1.575} = \frac{7}{9} \approx \mathbf{0.7778} \text{ } (\mathbf{77.78\%})$$
   *(Atau langsung dengan rumus $\frac{2 TP}{2 TP + FP + FN} = \frac{2(70)}{140 + 10 + 30} = \frac{140}{180} = \frac{7}{9} \approx 0.7778$).*

---

**2. Cara Menghitung ROC-AUC Tanpa Menggambar Kurva beserta Kompleksitasnya**

Secara probabilistik, **ROC-AUC** sama dengan peluang bahwa sebuah sampel kelas positif ($y=1$) yang dipilih secara acak memiliki skor prediksi lebih tinggi daripada sampel kelas negatif ($y=0$) yang dipilih secara acak:
$$\text{AUC} = P(s_+ > s_-) + \frac{1}{2} P(s_+ = s_-)$$

Nilai ini ekuivalen dengan **statistik uji Mann-Whitney U (Wilcoxon Rank-Sum)** yang dihitung melalui tahapan berikut tanpa perlu menghitung titik-titik $(FPR, TPR)$ atau menggambar kurva:

1. **Pengurutan dan Pemberian Peringkat (*Ranking*):**
   * Gabungkan seluruh $N = n_+ + n_-$ skor prediksi, lalu urutkan secara menaik dari skor terkecil hingga terbesar.
   * Berikan peringkat $r_i \in \{1, 2, \dots, N\}$. Jika terdapat skor kembar (*ties*) dari indeks $a$ hingga $b$, berikan **peringkat rata-rata** $\frac{a + b}{2}$ kepada masing-masing elemen yang kembar tersebut.
2. **Penjumlahan Peringkat Kelas Positif ($R_+$):**
   * Hitung jumlah peringkat untuk seluruh sampel yang berlabel aktual positif ($y_i = 1$):
     $$R_+ = \sum_{i : y_i = 1} r_i$$
3. **Konversi ke Nilai AUC:**
   * Kurangi $R_+$ dengan jumlah peringkat minimum yang mungkin diperoleh $n_+$ sampel positif ($\frac{n_+(n_+ + 1)}{2}$), lalu bagi dengan total pasangan positif-negatif ($n_+ \times n_-$):
     $$\text{AUC} = \frac{R_+ - \dfrac{n_+(n_+ + 1)}{2}}{n_+ \cdot n_-}$$

* **Analisis Kompleksitas Waktu dan Ruang:**
  * Pendekatan perbandingan berpasangan langsung (*brute-force pairwise comparison*) membutuhkan $\mathcal{O}(n_+ \cdot n_-) = \mathcal{O}(N^2)$.
  * Dengan metode **peringkat Mann-Whitney**, langkah pengurutan $N$ skor membutuhkan waktu **$\mathcal{O}(N \log N)$**, sedangkan pemberian peringkat rata-rata dan penjumlahan $R_+$ hanya membutuhkan satu kali pemindaian linier **$\mathcal{O}(N)$**.
  * Jadi, kompleksitas waktu totalnya adalah **$\mathcal{O}(N \log N)$** dengan kompleksitas ruang **$\mathcal{O}(N)$**.

### Soal A9 — Konsep Pemrograman untuk Sains Data *(4 poin: @1 poin per bagian | CPL-1 | CPMK-1 | Sub-CPMK-1, Sub-CPMK-2)*
> **Pertanyaan:**  
> **(a)** Apa itu fungsi murni dan idempoten, dan mengapa keduanya penting untuk fungsi pembersihan data?  
> **(b)** Kapan memakai `raise` dan kapan memakai `assert`?  
> **(c)** Tuliskan kerangka *custom exception* `DataTidakValid` beserta cara menangkapnya.  
> **(d)** Apa itu *closure*? Berikan contoh dari evaluasi model.

#### Jawaban A9:

---

#### **(a) Fungsi Murni (*Pure Function*) dan Idempoten (*Idempotent Function*)**

* **Fungsi Murni (*Pure Function*):** Fungsi yang (1) keluarannya hanya ditentukan sepenuhnya oleh argumen masukannya (*deterministik*), dan (2) tidak menimbulkan **efek samping (*side effects*)** apa pun terhadap lingkungan luar—khususnya tidak memodifikasi DataFrame atau array masukan secara *in-place* dan tidak mengubah variabel global.
* **Fungsi Idempoten (*Idempotent Function*):** Fungsi yang apabila diterapkan berulang kali secara beruntun pada hasil keluarannya sendiri akan tetap menghasilkan keluaran yang sama persis seperti penerapan pertama:
  $$f(f(x)) = f(x)$$
* **Mengapa Keduanya Penting pada Pembersihan Data?**
  1. *Keamanan Eksplorasi di Notebook (Reprodusibilitas):* Di Jupyter Notebook, sel kode sering dieksekusi ulang secara tidak berurutan. Jika fungsi pembersihan tidak murni (mengubah `df` secara *in-place*), maka dataset mentah (`hd`) akan ikut terubah sehingga eksperimen berikutnya kehilangan kondisi awal.
  2. *Ketahanan Pipeline:* Sifat idempoten menjamin bahwa apabila `bersihkan(df)` tidak sengaja terpanggil dua kali dalam pipeline (`bersihkan(bersihkan(df))`), fungsi tidak akan gagal (misalnya error karena kolom `age` sudah berubah menjadi `float` dan tidak lagi memiliki metode `.str`), serta tidak akan menggeser nilai median atau menghapus baris tambahan.

---

#### **(b) Kapan Memakai `raise` dan Kapan Memakai `assert`?**

| Aspek | `raise` (Eksepsi Eksplisit) | `assert` (Asersi Internal) |
| :--- | :--- | :--- |
| **Tujuan Utama** | Menegakkan **kontrak API publik** dan menangani kesalahan masukan atau kondisi *runtime* yang tidak valid. | Memverifikasi **invarians internal (*sanity check* / *post-condition*)** selama pengembangan dan pengujian. |
| **Contoh Penggunaan** | Melempar `ValueError` saat kunci tabel kanan duplikat pada `gabung_aman`, atau melempar `RuntimeError` saat `transform()` dipanggil sebelum `fit()`. | Di baris akhir `bersihkan(df)`: `assert df_bersih.isna().sum().sum() == 0` untuk memastikan logika imputasi kita tidak menyisakan `NaN`. |
| **Perilaku di Produksi** | **Selalu dieksekusi** dalam kondisi apa pun; kita dapat memilih tipe eksepsi yang spesifik (`ValueError`, `KeyError`, `RuntimeError`). | **Dapat dinonaktifkan** oleh interpreter Python jika dijalankan dengan bendera optimasi (`python -O`), dan hanya melempar `AssertionError`. |

> **Aturan praktis:** Gunakan `raise` untuk memvalidasi argumen dari luar/pengguna; gunakan `assert` untuk memverifikasi kebenaran logika internal kode kita sendiri.

---

#### **(c) Kerangka *Custom Exception* `DataTidakValid` Beserta Cara Menangkapnya**

```python
class DataTidakValid(ValueError):
    """Eksepsi khusus ketika struktur atau nilai DataFrame melanggar aturan kualitas data."""
    def __init__(self, pesan, kolom=None, detail=None):
        self.kolom = kolom
        self.detail = detail
        info_kolom = f" [Kolom: '{kolom}']" if kolom else ""
        super().__init__(f"{pesan}{info_kolom}")


def validasi_kolesterol(df):
    """Contoh fungsi yang melempar DataTidakValid."""
    if 'chol' not in df.columns:
        raise DataTidakValid("Kolom wajib tidak ditemukan", kolom='chol')
    if (df['chol'] < 0).any():
        raise DataTidakValid("Ditemukan nilai kolesterol negatif", kolom='chol')
    return True


# Cara menangkap (handling) custom exception DataTidakValid:
try:
    df_uji = pd.DataFrame({'chol': [210, -5, 190]})
    validasi_kolesterol(df_uji)
except DataTidakValid as err:
    print(f"Tertangkap galat kualitas data -> {err} (pada kolom: {err.kolom})")
```

---

#### **(d) Pengertian *Closure* dan Contohnya pada Evaluasi Model**

* **Pengertian *Closure*:** *Closure* adalah sebuah fungsi dalam (*inner function*) yang dikembalikan oleh fungsi pembungkus (*enclosing/outer function*) dan **tetap mengingat (mengikat) variabel-variabel lokal dari ruang lingkup (*scope*) fungsi pembungkusnya**, meskipun fungsi pembungkus tersebut sudah selesai dieksekusi.
* **Contoh pada Evaluasi Model:**
  Ketika melakukan validasi silang (*cross-validation*), antarmuka evaluator biasanya hanya menerima fungsi dengan signature `skor(y_true, y_prob)`. Dengan *closure*, kita dapat menyuntikkan parameter konfigurasi (seperti ambang batas klasifikasi `ambang` atau bobot `beta` pada $F_\beta$-score) ke dalam fungsi evaluator:

```python
def buat_evaluator_fbeta(ambang=0.5, beta=2.0):
    """Outer function yang mengonfigurasi ambang dan bobot beta (mengembalikan closure)."""
    beta_kuadrat = beta ** 2  # variabel bebas (free variable) yang disimpan di dalam closure

    def hitung_skor(y_true, y_prob):
        """Inner function (closure) dengan signature standar (y_true, y_prob)."""
        y_true_arr = np.asarray(y_true)
        y_pred = (np.asarray(y_prob) >= ambang).astype(int)

        tp = np.sum((y_true_arr == 1) & (y_pred == 1))
        fp = np.sum((y_true_arr == 0) & (y_pred == 1))
        fn = np.sum((y_true_arr == 1) & (y_pred == 0))

        presisi = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        penyebut = (beta_kuadrat * presisi) + recall
        return ((1 + beta_kuadrat) * presisi * recall / penyebut) if penyebut > 0 else 0.0

    return hitung_skor


# Contoh pemakaian closure untuk skrining medis (menekankan Recall dengan beta=2, ambang=0.35):
skor_skrining = buat_evaluator_fbeta(ambang=0.35, beta=2.0)
# Fungsi skor_skrining kini cukup dipanggil dengan (y_true, y_prob) saja:
# nilai_f2 = skor_skrining(y_val, prob_val)
```